In [1]:
!nvidia-smi

Mon Sep 14 10:51:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:

!pip install -q -U transformers datasets peft trl bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 14.6 MB/s eta 0:00:00


In [3]:

import torch
import transformers
import datasets
import peft #Parameter-Efficient Fine-Tuning
import trl  #Transformer Reinforcement Learning, It provides high-level tools like
            #the SFTTrainer (Supervised Fine-Tuning Trainer) to easily combine the dataset,
            #  the compressed model, and the PEFT adapters into a streamlined training loop
import bitsandbytes #Quantization, It compresses large 16-bit or 32-bit models down into 8-bit or 4-bit sizes
import accelerate

print("=" * 60)
print("ENVIRONMENT")
print("=" * 60)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("Accelerate:", accelerate.__version__)

print("\nCUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. In Colab, select a GPU runtime."
    )

print("GPU:", torch.cuda.get_device_name(0))

total_vram = (
    torch.cuda.get_device_properties(0).total_memory
    / (1024 ** 3)
)

print(f"Total VRAM: {total_vram:.2f} GB")

free_vram, total_vram = torch.cuda.mem_get_info()

print(f"Free VRAM: {free_vram / (1024 ** 3):.2f} GB")
print(f"Total VRAM: {total_vram / (1024 ** 3):.2f} GB")

!nvidia-smi

ENVIRONMENT
PyTorch: 2.11.0+cu128
Transformers: 5.17.0
Datasets: 5.0.1
PEFT: 0.20.0
TRL: 1.13.0
BitsAndBytes: 0.50.2
Accelerate: 1.15.0

CUDA available: True
CUDA version: 12.8
GPU: Tesla T4
Total VRAM: 14.56 GB
Free VRAM: 14.46 GB
Total VRAM: 14.56 GB
Mon Sep 14 10:52:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |   

In [4]:
import torch
from transformers import AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# Load the tokenizer for the model
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Setting up the model with 4-bit quantization using BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("4-bit compute dtype:", bnb_config.bnb_4bit_compute_dtype)

print(f"model name: {model_name}")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

4-bit compute dtype: torch.float16
model name: Qwen/Qwen2.5-1.5B-Instruct


In [5]:
from transformers import AutoModelForCausalLM

qlora_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto"
)
print(f"qlora model: {qlora_model}")
print(f"qlora model device: {qlora_model.device}")

free_vram, total_vram  = torch.cuda.mem_get_info()
print(f"Free VRAM: {free_vram / (1024 ** 3):.2f} GB")
print(f"Total VRAM: {total_vram / (1024 ** 3):.2f} GB")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

qlora model: Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qw

In [6]:
total_par = sum(
    p.numel()
    for p in qlora_model.parameters()
)

print("\nModel parameters:")
print(f"Total parameters: {total_par:,}")
print(f"Total parameters: {total_par / 1e9:.2f}B")

# Check how much GPU memory the 4-bit model uses
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / (1024 ** 3)
    reserved = torch.cuda.memory_reserved() / (1024 ** 3)

    print("\nQLoRA 4-bit GPU memory:")
    print(f"Allocated: {allocated:.2f} GB")
    print(f"Reserved:  {reserved:.2f} GB")


Model parameters:
Total parameters: 888,616,448
Total parameters: 0.89B

QLoRA 4-bit GPU memory:
Allocated: 1.07 GB
Reserved:  1.12 GB


In [7]:
# Check the actual model configuration

print("Model name:", model_name)
print("Architecture:", qlora_model.config.architectures)
print("Hidden size:", qlora_model.config.hidden_size)
print("Number of layers:", qlora_model.config.num_hidden_layers)
print("Vocab size:", qlora_model.config.vocab_size)

print("\nParameter count:")
print(sum(p.numel() for p in qlora_model.parameters()))
print(qlora_model)

Model name: Qwen/Qwen2.5-1.5B-Instruct
Architecture: ['Qwen2ForCausalLM']
Hidden size: 1536
Number of layers: 28
Vocab size: 151936

Parameter count:
888616448
Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (i

In [8]:
from peft import LoraConfig, get_peft_model
from peft import prepare_model_for_kbit_training

#prepare the model for k-bit trainning

qLora_base_model = prepare_model_for_kbit_training(qlora_model)

#lora adopter config

lora_config = LoraConfig(
    r=16, # rank is the number of parameters in the low-rank matrices used to approximate the original weight matrices. A higher rank allows for a more paramter trainable (expressive) model, but also increases the number of parameters and computational cost.
    lora_alpha = 32, # Lora_alpha is a hyperparameter that controls the scalibility of the low-rank matrices
    lora_dropout = 0.05, # Lora_dropout is a regularization technique that randomly sets a fraction of the low-rank matrix elements to zero during training, which helps prevent overfitting

    target_modules =[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    bias = "none",
    task_type = "CAUSAL_LM"
)



In [9]:
#adding lora adopter to the 4-bit model

qlora_model = get_peft_model(
    qLora_base_model,
    lora_config
)

In [10]:
# baba blackship nah just kidding let's see the trainable parameters of the model
qlora_model.print_trainable_parameters()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [11]:

from datasets import load_dataset, Dataset

dataset = load_dataset("raghu298/ml-interview-sft-dataset")

print("Original dataset:")
print(dataset)

# checking for duplicate questions
questions = [
    example["question"].strip().lower()
    for example in dataset["train"]
]

print("\nOriginal examples:", len(questions))
print("Unique questions:", len(set(questions)))
print("Duplicate questions:", len(questions) - len(set(questions)))


#

README.md:   0%|          | 0.00/5.31k [00:00<?, ?B/s]

ml_interview_sft.jsonl:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/566 [00:00<?, ? examples/s]

Original dataset:
DatasetDict({
    train: Dataset({
        features: ['messages', 'question', 'answer', 'topic', 'source'],
        num_rows: 566
    })
})

Original examples: 566
Unique questions: 531
Duplicate questions: 35


In [12]:

unique_data = []
seen_questions = set()

for example in dataset["train"]:
    question = example["question"].strip().lower()

    if question not in seen_questions:
        seen_questions.add(question)
        unique_data.append(example)

clean_dataset = Dataset.from_list(unique_data)

print("\nAfter deduplication:", len(clean_dataset))
print("Removed:", len(dataset["train"]) - len(clean_dataset))


After deduplication: 531
Removed: 35


In [13]:

split_1 = clean_dataset.train_test_split(
    test_size=0.10,
    seed=42
)

train_val = split_1["train"]
test_dataset = split_1["test"]

#    10% of the original dataset ≈ validation
split_2 = train_val.train_test_split(
    test_size=0.1111,
    seed=42
)

train_dataset = split_2["train"]
val_dataset = split_2["test"]

# just verifying
print("\nFinal dataset:")
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

print("\nExample:")
print(train_dataset[0])


Final dataset:
Train: 424
Validation: 53
Test: 54

Example:
{'messages': [{'content': 'You are an expert AI/ML interview coach. You provide clear, detailed, and technically accurate answers to machine learning, deep learning, NLP, and AI engineering interview questions. Include code examples when relevant. Keep answers concise but thorough.', 'role': 'system'}, {'content': "Your seq2seq model's decoder outputs irrelevant content even though encoder embeddings look correct. \nWhere would you trace the fault?", 'role': 'user'}, {'content': 'The problem is likely in cross-attention between encoder and decoder. Check if the decoder is actually \nusing encoder information. \nDebugging steps: Visualize cross-attention weights. They should focus on relevant encoder positions. If \ncross-attention is uniform or focuses on wrong tokens, that\'s your issue. Check if cross-attention is getting gradients \nduring training. \nFor example, translating "The red car" to French should have cross-atten

In [14]:
#setting up the data for sft 🫂

def format_example(example):
    return{
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize = False
        )
    }

train_dataset = train_dataset.map(format_example)
val_dataset = val_dataset.map(format_example)
test_dataset = test_dataset.map(format_example)

print("Formatted dataset:")
print(train_dataset[0]["text"])

Map:   0%|          | 0/424 [00:00<?, ? examples/s]

Map:   0%|          | 0/53 [00:00<?, ? examples/s]

Map:   0%|          | 0/54 [00:00<?, ? examples/s]

Formatted dataset:
<|im_start|>system
You are an expert AI/ML interview coach. You provide clear, detailed, and technically accurate answers to machine learning, deep learning, NLP, and AI engineering interview questions. Include code examples when relevant. Keep answers concise but thorough.<|im_end|>
<|im_start|>user
Your seq2seq model's decoder outputs irrelevant content even though encoder embeddings look correct. 
Where would you trace the fault?<|im_end|>
<|im_start|>assistant
The problem is likely in cross-attention between encoder and decoder. Check if the decoder is actually 
using encoder information. 
Debugging steps: Visualize cross-attention weights. They should focus on relevant encoder positions. If 
cross-attention is uniform or focuses on wrong tokens, that's your issue. Check if cross-attention is getting gradients 
during training. 
For example, translating "The red car" to French should have cross-attention focusing on "red" when generating 
"rouge". If it focuses o

In [15]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="/content/qlora-interview-coach",

    num_train_epochs=3,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=2,

    learning_rate=2e-4,

    logging_steps=10,

    eval_strategy="steps",
    eval_steps=50,

    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,

    fp16=False,
    bf16=False,

    gradient_checkpointing=False,

    report_to="none",

    max_length=512,
    dataset_text_field="text"
)

In [16]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=qlora_model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    processing_class=tokenizer
)

print("QLoRA trainer recreated successfully.")

Tokenizing train dataset:   0%|          | 0/424 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/424 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/424 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/424 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/53 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/53 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/53 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/53 [00:00<?, ? examples/s]

QLoRA trainer recreated successfully.


In [17]:

print(
    "QLoRA compute dtype:",
    qlora_model.config.quantization_config.bnb_4bit_compute_dtype
)

QLoRA compute dtype: torch.float16


In [18]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,1.476965,1.535002,1.504529,110554.000000,0.676125
100,1.246838,1.438406,1.364023,220188.000000,0.693470
150,1.331694,1.416435,1.337394,331812.000000,0.696871
159,1.331694,1.416291,1.335628,350526.000000,0.696768


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=159, training_loss=1.4331235765661083, metrics={'train_runtime': 699.8942, 'train_samples_per_second': 1.817, 'train_steps_per_second': 0.227, 'total_flos': 3767747824201728.0, 'train_loss': 1.4331235765661083, 'epoch': 3.0})

In [19]:
qlora_model.save_pretrained("/content/ai-interview-coach-qlora")
tokenizer.save_pretrained("/content/ai-interview-coach-qlora")

('/content/ai-interview-coach-qlora/tokenizer_config.json',
 '/content/ai-interview-coach-qlora/chat_template.jinja',
 '/content/ai-interview-coach-qlora/tokenizer.json')

In [20]:
# test_loss and perplexity just tell how well the next token is produced by comparing the test data set

import math

print("QLORA TEST EVALUATION")


eval_results = trainer.evaluate(
    eval_dataset=test_dataset
)

test_loss = eval_results["eval_loss"]
test_perplexity = math.exp(test_loss)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Perplexity: {test_perplexity:.2f}")

QLORA TEST EVALUATION


Tokenizing eval dataset:   0%|          | 0/54 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/54 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/54 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/54 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Step,Entropy,Num Tokens,Mean Token Accuracy
1.331694,1.393291,159,1.314487,350526.000000,0.707758


Test Loss: 1.3933
Test Perplexity: 4.03


In [21]:
# compring the base model and the qlora model
def generate_answer(model, tokenizer, question, max_new_tokens=150):

    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert AI/ML interview coach. "
                "Provide a clear, technically accurate interview answer."
            )
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )


print("=" * 60)
print("BASE vs QLORA")
print("=" * 60)

for i, example in enumerate(test_dataset.select(range(3))):

    question = example["question"]

    with qlora_model.disable_adapter():
        base_answer = generate_answer(
            qlora_model,
            tokenizer,
            question
        )

    qlora_answer = generate_answer(
        qlora_model,
        tokenizer,
        question
    )

    print("\n" + "=" * 80)
    print(f"QUESTION {i+1}:")
    print(question)

    print("\nBASE:")
    print(base_answer)

    print("\nQLORA:")
    print(qlora_answer)

BASE vs QLORA

QUESTION 1:
Explain Reducing hallucinations in the context of vector databases.

BASE:
Reducing hallucinations in the context of vector databases involves enhancing the accuracy and reliability of search results by minimizing false positives or "hallucinations" that occur during the indexing and querying process. Vector databases leverage high-dimensional numerical vectors to represent data points, enabling efficient similarity searches through distance metrics like Euclidean or cosine distances.

The challenge lies in ensuring that when a query is executed against these vectors, it retrieves relevant documents rather than spurious matches due to incorrect or misleading representations of the input data. This can be achieved through several strategies:

1. **Data Preprocessing**: Cleaning and normalizing the input data before indexing helps reduce noise and inconsistencies that might lead to hallucinations. Techniques such as normalization (e.g., scaling values), removin

In [22]:
!pip install -q evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.2 MB/s eta 0:00:00


In [23]:
import evaluate

rouge = evaluate.load("rouge")
predictions_base =[]
prediction_qlora =[]
references =[]

eval_samples =  test_dataset.select(
    range(min(20, len(test_dataset)))
)

for example in eval_samples:
    question = example["question"]
    reference_answer = example["answer"]

    with qlora_model.disable_adapter():
        base_answer = generate_answer(
            qlora_model,
            tokenizer,
            question
        )

    qlora_answer = generate_answer(
        qlora_model,
        tokenizer,
        question
    )

    predictions_base.append(base_answer)
    prediction_qlora.append(qlora_answer)
    references.append(reference_answer)

base_rouge = rouge.compute(
    predictions=predictions_base,
    references=references
)

qlora_rouge = rouge.compute(
    predictions=prediction_qlora,
    references=references
)
print("ROUGE-L COMPARISON")
print(f"Base ROUGE-L:  {base_rouge['rougeL']:.4f}")
print(f"QLoRA ROUGE-L: {qlora_rouge['rougeL']:.4f}")

delta = qlora_rouge["rougeL"] - base_rouge["rougeL"]

print(f"ROUGE-L change: {delta:+.4f}")

ROUGE-L COMPARISON
Base ROUGE-L:  0.1213
QLoRA ROUGE-L: 0.2274
ROUGE-L change: +0.1061


In [24]:
import time

def benchmark_qlora(
    model,
    tokenizer,
    dataset,
    n_samples=10,
    max_new_tokens=100
):

    model.eval()

    times = []

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    samples = dataset.select(
        range(min(n_samples, len(dataset)))
    )

    for example in samples:

        messages = [
            {
                "role": "system",
                "content": (
                    "You are an expert AI/ML interview coach. "
                    "Provide a clear, technically accurate interview answer."
                )
            },
            {
                "role": "user",
                "content": example["question"]
            }
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt"
        ).to(model.device)

        torch.cuda.synchronize()

        start = time.perf_counter()

        with torch.no_grad():
            model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False
            )

        torch.cuda.synchronize()

        end = time.perf_counter()

        times.append(end - start)

    avg_time = sum(times) / len(times)

    peak_vram = (
        torch.cuda.max_memory_allocated()
        / (1024 ** 3)
    )

    return avg_time, 1 / avg_time, peak_vram


avg_time, throughput, peak_vram = benchmark_qlora(
    qlora_model,
    tokenizer,
    test_dataset
)

print("=" * 60)
print("QLORA EFFICIENCY")
print("=" * 60)

print(f"Average inference time: {avg_time:.2f} sec")
print(f"Throughput: {throughput:.4f} samples/sec")
print(f"Peak VRAM: {peak_vram:.2f} GB")

QLORA EFFICIENCY
Average inference time: 12.36 sec
Throughput: 0.0809 samples/sec
Peak VRAM: 1.60 GB


In [25]:

from transformers import GenerationConfig, ContinuousBatchingConfig
import time
import torch


def benchmark_continuous_batching(
    model,
    tokenizer,
    dataset,
    n_requests=6,
    max_new_tokens=100
):

    model.eval()


    samples = dataset.select(
        range(min(n_requests, len(dataset)))
    )

    prompts = []

    for example in samples:

        messages = [
            {
                "role": "system",
                "content": (
                    "You are an expert AI/ML interview coach. "
                    "Provide a clear, technically accurate interview answer."
                )
            },
            {
                "role": "user",
                "content": example["question"]
            }
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # Convert prompt to token IDs
        input_ids = tokenizer.encode(
            prompt,
            add_special_tokens=False
        )

        prompts.append(input_ids)

    # Generation configuration

    generation_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )

    # Continuous batching configuration

    continuous_batching_config = ContinuousBatchingConfig(
    max_memory_percent=0.80,
    block_size=256,
    scheduler_type="prefill_first",
    max_requests_per_batch=n_requests,
    use_cuda_graph=True,
    use_async_batching=True,
    allow_block_sharing=True
    )


    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    torch.cuda.synchronize()

    start = time.perf_counter()

#c_b
    outputs = model.generate_batch(
    prompts,
    generation_config=generation_config,
    continuous_batching_config=continuous_batching_config
    )

    torch.cuda.synchronize()

    end = time.perf_counter()
#metrices
    total_time = end - start

    peak_vram = (
        torch.cuda.max_memory_allocated()
        / (1024 ** 3)
    )

    throughput = len(prompts) / total_time

    average_time = total_time / len(prompts)
    print("CONTINUOUS BATCHING")

    print(f"Requests:              {len(prompts)}")
    print(f"Total time:            {total_time:.2f} sec")
    print(f"Average time/request:  {average_time:.2f} sec")
    print(f"Throughput:            {throughput:.4f} requests/sec")
    print(f"Peak VRAM:             {peak_vram:.2f} GB")

    return outputs

In [26]:

outputs = benchmark_continuous_batching(
    qlora_model,
    tokenizer,
    test_dataset,
    n_requests=500,
    max_new_tokens=100
)

CONTINUOUS BATCHING
Requests:              54
Total time:            102.93 sec
Average time/request:  1.91 sec
Throughput:            0.5246 requests/sec
Peak VRAM:             10.47 GB


In [27]:
from datasets import concatenate_datasets

benchmark_dataset = concatenate_datasets(
    [test_dataset] * 19
).shuffle(seed=42).select(range(1000))

print(len(benchmark_dataset))

1000


In [28]:
for n in [100, 500, 750, 1000]:

    print(f"TEST: {n} REQUESTS")

    benchmark_continuous_batching(
        qlora_model,
        tokenizer,
        benchmark_dataset,
        n_requests=n,
        max_new_tokens=100
    )

TEST: 100 REQUESTS
CONTINUOUS BATCHING
Requests:              100
Total time:            113.23 sec
Average time/request:  1.13 sec
Throughput:            0.8831 requests/sec
Peak VRAM:             10.39 GB
TEST: 500 REQUESTS
CONTINUOUS BATCHING
Requests:              500
Total time:            350.71 sec
Average time/request:  0.70 sec
Throughput:            1.4257 requests/sec
Peak VRAM:             10.37 GB
TEST: 750 REQUESTS
CONTINUOUS BATCHING
Requests:              750
Total time:            487.27 sec
Average time/request:  0.65 sec
Throughput:            1.5392 requests/sec
Peak VRAM:             9.97 GB
TEST: 1000 REQUESTS
CONTINUOUS BATCHING
Requests:              1000
Total time:            632.25 sec
Average time/request:  0.63 sec
Throughput:            1.5817 requests/sec
Peak VRAM:             9.76 GB


In [29]:
def generate_question(model, tokenizer, topic="RAG", difficulty="medium"):

    prompt = f"""
You are conducting a technical interview for a fresher AI Engineer.

Topic: {topic}
Difficulty: {difficulty}

Generate EXACTLY ONE interview question.

Rules:
- Ask only ONE question.
- Do not ask multiple questions.
- Do not add a second question.
- Do not provide an answer.
- Do not provide an explanation.
- Do not use headings.
- Do not use bullet points.
- The question must be clear and technically relevant.
- The question should be appropriate for a fresher AI Engineer.
- Return ONLY the question.
"""

    messages = [
        {
            "role": "system",
            "content": "You are a professional AI Engineer interviewer."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            use_cache=True
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    question = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    # Remove markdown formatting
    question = question.replace("**", "").strip()

    return question

In [30]:
def evaluate_answer(model, tokenizer, question, answer):

    # Handle non-answers directly
    non_answers = {
        "dk",
        "don't know",
        "dont know",
        "i don't know",
        "i dont know",
        "idk",
        "no idea",
        "not sure",
        "i'm not sure",
        "im not sure",
        "skip",
        "pass"
    }

    if answer.lower().strip() in non_answers:

        return """SCORE: 0/10

FEEDBACK:
You did not attempt the question.

WHAT WAS GOOD:
No significant technical points were provided.

WHAT WAS MISSING:
You needed to provide a technical explanation related to the question.

IDEAL ANSWER:
Try to explain the main concept first, then describe how it works and give a simple example if possible.

LEARNING POINT:
When you don't know an interview question, try to explain any related concept you know instead of immediately giving up.
"""

    prompt = f"""
You are a strict but fair AI Engineer interviewer evaluating a fresher.

INTERVIEW QUESTION:
{question}

CANDIDATE ANSWER:
{answer}

Evaluate ONLY the candidate's answer to the interview question.

Important rules:
- Do not assume the candidate said something they did not say.
- Do not give credit for information that is not present in the answer.
- Give 0/10 if the answer is completely irrelevant.
- Give a low score if the answer is incomplete.
- Give a high score only when the answer is technically correct, relevant, and sufficiently complete.
- Do not be overly generous.
- Do not invent strengths.

Return EXACTLY this format:

SCORE: X/10

FEEDBACK:
Give concise and honest feedback.

WHAT WAS GOOD:
Mention only technically correct points actually present in the candidate's answer.

WHAT WAS MISSING:
Mention important concepts that were missing.

IDEAL ANSWER:
Give a concise, technically accurate answer suitable for a fresher AI Engineer interview.

LEARNING POINT:
Teach ONE important concept from this question in simple terms.
"""

    messages = [
        {
            "role": "system",
            "content": (
                "You are a strict but helpful AI Engineer interviewer. "
                "Evaluate answers honestly and accurately."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=350,
            do_sample=False,
            use_cache=True
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    evaluation = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return evaluation

In [ ]:
def interview_session(model, tokenizer, topic="AI/ML", difficulty="medium"):
    question_number = 1

    print("=" * 60)
    print("🎤 AI INTERVIEW COACH")
    print("=" * 60)
    print(f"Topic: {topic}")
    print(f"Difficulty: {difficulty}")
    print("Type 'bye' anytime to end the interview.")
    print()

    while True:

        # Generate interview question
        question = generate_question(
            model,
            tokenizer,
            topic=topic,
            difficulty=difficulty
        )

        print(f"INTERVIEWER — Question {question_number}:")
        print(question)
        print()

        # Get candidate answer
        answer = input("YOUR ANSWER: ")

        # End session
        if answer.strip().lower() == "bye":
            print("\nInterview session ended.")
            break

        # Evaluate answer
        evaluation = evaluate_answer(
            model,
            tokenizer,
            question,
            answer
        )

        print("\n" + "-" * 60)
        print("FEEDBACK")
        print("-" * 60)
        print(evaluation)
        print("-" * 60)

        question_number += 1
        print()

In [ ]:
interview_session(
    qlora_model,
    tokenizer,
    topic="RAG",
    difficulty="medium"
)